# Урок 23. Решение задач на определение влажности воздуха

Все задачи на влажность решаются по одной схеме, и в ней всего два инструмента: таблица
плотности насыщенного пара и формула $\varphi = \rho / \rho_0 \cdot 100\,\%$. К ним добавляется
психрометрическая таблица для задач с двумя термометрами.

## 1. Схема решения

1. Определите, что дано: температура воздуха, абсолютная влажность, относительная влажность,
   точка росы или показания психрометра.
2. Найдите в таблице плотность насыщенного пара $\rho_0$ при температуре воздуха. Если
   температуры нет в таблице, возьмите значение между соседними.
3. Если известна точка росы $t_{\text{р}}$, абсолютная влажность равна $\rho_0(t_{\text{р}})$:
   при этой температуре пар как раз насыщен.
4. Свяжите величины формулой $\varphi = \rho / \rho_0$ и найдите неизвестное.
5. Вопрос «выпадет ли роса, иней, туман» сводится к сравнению: если при новой температуре
   $\rho_0$ окажется меньше $\rho$, лишний пар сконденсируется.

Функции ниже делают то же, что таблица, только без поиска глазами по строкам.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phys import style

style.use()
T  = np.array([-10, -5, 0, 5, 10, 15, 20, 25, 30, 50])
R0 = np.array([2.1, 3.2, 4.8, 6.8, 9.4, 12.8, 17.3, 23.0, 30.4, 83.0])
rho0 = lambda t: float(np.interp(t, T, R0))
tochka_rosy = lambda rho: float(np.interp(rho, R0, T))

tt = np.linspace(-10, 30, 200)
fig, ax = style.axes("Температура, °C", "Плотность насыщенного пара, г/м³",
                     "Чем теплее воздух, тем больше пара он может удержать")
ax.plot(tt, [rho0(t) for t in tt])
ax.plot(T[:-1], R0[:-1], "o")
plt.show()
print(f"от 0 до 30 °C ёмкость воздуха по пару выросла в {rho0(30) / rho0(0):.1f} раза")

Кривая растёт всё круче — каждые 10 °C почти удваивают предельное содержание пара.
Отсюда и все погодные следствия: тёплый воздух, остывая, «выжимает» из себя воду.

## 2. Разбор задач

**Марон, № 270.** Какой воздух кажется суше: с 5 г/м³ пара при 30 °C или с 1 г/м³ при 0 °C?

In [ ]:
for rho, t in [(5, 30), (1, 0)]:
    print(f"{rho} г/м³ при {t:2d} °C: φ = {rho / rho0(t) * 100:.0f} %")

Суше первый: пара в нём в пять раз больше, но относительная влажность ниже. Так зимой
в квартире с батареями: воздух с улицы, нагретый до 22 °C, имеет влажность 15–20 %.

**Марон, № 271.** При 2 °C относительная влажность 60 %. Появится ли ночью иней, если
температура упадёт до −3 °C?

In [ ]:
rho = 0.6 * rho0(2)
print(f"пара в воздухе: {rho:.2f} г/м³; насыщение при −3 °C наступает при {rho0(-3):.2f} г/м³")
print("иней", "появится" if rho > rho0(-3) else "не появится: пар ещё не насыщен")

**Марон, № 272.** Над морем при 25 °C влажность 95 %. При какой температуре появится туман?

In [ ]:
from phys import quiz
quiz.numeric("Температура появления тумана, °C:", answer=24, unit="°C", tol=0.05,
             explain="ρ = 0,95 · 23,0 ≈ 21,9 г/м³. Туман появится, когда эта плотность станет насыщающей — "
                     "по таблице между 20 °C (17,3) и 25 °C (23,0), около 24 °C. Достаточно охлаждения "
                     "на один градус. Ответ сборника — около 24 °C.")

**Марон, № 275.** В подвале при 8 °C влажность 100 %. На сколько градусов надо прогреть воздух,
чтобы влажность стала 60 %?

In [ ]:
quiz.numeric("На сколько градусов повысить температуру:", answer=8, unit="°C", tol=0.1,
             explain="Пара в подвале ρ₀(8 °C) ≈ 8,4 г/м³. Для 60 % нужно, чтобы насыщающая плотность была "
                     "8,4 / 0,6 = 14 г/м³ — это около 16 °C. Значит, прогреть на 8 °C. Ответ сборника — на 8 °C.")

**Марон, № 274.** В комнате 18 °C и влажность 56 %. Что показывает влажный термометр
психрометра?

In [ ]:
quiz.numeric("Показание влажного термометра, °C:", answer=13, unit="°C", tol=0.05,
             explain="Строка 18 °C психрометрической таблицы: 100, 91, 82, 73, 65, 56 — влажность 56 % "
                     "стоит в столбце разности 5 °C. Влажный термометр показывает 18 − 5 = 13 °C. "
                     "Ответ сборника — 13 °C.")

**Марон, № 273.** В теплице для проращивания семян нужны 30 °C и влажность 90 %. Сухой
термометр показывает 30 °C, влажный — 29 °C. Выполнено ли требование?

In [ ]:
quiz.reveal("Показать разбор",
            "Разность показаний 1 °C. В строках таблицы для 24 и 26 °C разности в один градус "
            "соответствует влажность 92 %; для 30 °C она будет ещё чуть выше. Это больше 90 % — "
            "требование выполнено. Ответ сборника: выполняется.")

## 3. Ваши данные

Впишите свои показания — расчёт даст влажность, точку росы и предупредит о росе.

In [ ]:
# ↓ измерения: температура воздуха и относительная влажность (например, с домашнего гигрометра)
t_vozduha = 22.0
phi = 45.0                      # %
t_nochyu = 8.0                  # до какой температуры остынет воздух ночью

rho = phi / 100 * rho0(t_vozduha)
print(f"пара в воздухе: {rho:.1f} г/м³ (насыщение при {t_vozduha:.0f} °C — {rho0(t_vozduha):.1f} г/м³)")
print(f"точка росы: {tochka_rosy(rho):.1f} °C")
if rho > rho0(t_nochyu):
    print(f"при {t_nochyu:.0f} °C выпадет роса: лишние {rho - rho0(t_nochyu):.1f} г с каждого кубометра")
else:
    print(f"при {t_nochyu:.0f} °C росы не будет, влажность станет {rho / rho0(t_nochyu) * 100:.0f} %")

## 4. Задача на оценку

**Сколько воды выпадет за ночь росой с гектара луга**, если вечером при 20 °C влажность
была 80 %, а к утру приземный слой воздуха толщиной 2 м остыл до 10 °C?

In [ ]:
quiz.numeric("Масса росы с гектара, кг:", answer=91, unit="кг", tol=0.15,
             explain="Вечером в кубометре 0,8 · 17,3 ≈ 13,8 г пара. Утром при 10 °C удержится только 9,4 г; "
                     "лишние 4,4 г с кубометра выпадут росой. Объём слоя над гектаром: 10 000 м² · 2 м = "
                     "20 000 м³, роса — около 90 кг, то есть 90 литров на гектар. Настоящая роса обильнее: "
                     "пар подтягивается и из более высоких слоёв, но порядок величины верен.")

## 5. Проверьте себя

In [ ]:
quiz.choice("Точка росы — это...",
    ["температура, при которой пар в воздухе становится насыщенным", "температура воздуха утром",
     "температура кипения воды при данном давлении", "количество росы в граммах"],
    correct=0,
    explain="Охлаждая воздух, мы уменьшаем ρ₀; когда она сравняется с имеющейся ρ, пар насыщен — "
            "это и есть точка росы. Дальнейшее охлаждение вызывает конденсацию.")

In [ ]:
quiz.numeric("При 20 °C влажность 50 %. Сколько пара в кубометре? Ответ в г/м³.", answer=8.65, unit="г/м³", tol=0.03,
             explain="ρ = φ · ρ₀ = 0,5 · 17,3 = 8,65 г/м³.")

## 6. Домашнее задание

1. **Учебник:** § 20, задание 13 — решите письменно.
2. **Марон, № 270, 271** — оформите решение с расчётом, как в разборе.
3. **Сравните** влажность в комнате при закрытом и открытом окне: измерьте утром и вечером
   психрометром или домашним гигрометром, объясните разницу.
4. **Со звёздочкой.** Днём при 25 °C влажность 60 %. Комнату проветрили и закрыли, ночью
   в ней стало 15 °C. Запотеют ли окна? Какая влажность установится?

In [ ]:
quiz.reveal("Показать ответ к задаче со звёздочкой",
            "Пара в комнате 0,6 · 23,0 = 13,8 г/м³. При 15 °C насыщающая плотность 12,8 г/м³ — меньше, "
            "чем есть. Окна запотеют, а лишний 1 г с кубометра осядет на стёклах; влажность установится "
            "100 %. Если бы ночью было 18 °C (ρ₀ ≈ 15,4), окна остались бы сухими при влажности около 90 %.")

```{admonition} Посмотреть
:class: seealso
* **GetAClass. Влажность воздуха**: [RuTube](https://rutube.ru/video/b6d32d58de22bd40947353a1bed3e6d8/)
* **Павел Виктор. Уроки 127–129 (осн). Задачи на тепловой баланс и комбинированные задачи** —
  для повторения перед контрольной: [плейлист на RuTube](https://rutube.ru/plst/449395/)
```

На следующем уроке — тепловые двигатели: как заставить тепло совершать работу.